In [ ]:

import serial
import time
import re

import random
import numpy as np

# Test connection
arduino = serial.Serial(port='COM3', baudrate=115200, timeout=1)
time.sleep(2)
print(f"Connected to: {arduino.port}")
print(f"Is open: {arduino.is_open}")

#training stuff
forward = False


#state space and action space definition
primary_arm_discrete_positions = 7  #clipped between 15 and 90 degree . Allowed positions: [2, 3, 4, 6, 7, 10, 11, 16, 19, 31, 46, 91]
secondary_arm_discete_positions = 10  #clipped between 20 and 170 degree. Allowed positions: [2, 3, 4, 5, 6, 7, 10, 11, 13, 16, 19, 21, 31, 37, 46, 61, 91, 181]
total_actions = primary_arm_discrete_positions + secondary_arm_discete_positions
action = None



# Q table defined as [layer][row][col]
Q = np.zeros((total_actions, secondary_arm_discete_positions, primary_arm_discrete_positions))

discount_factor = 0.9 # discount factor
learn_rate = 0.01 #learn rate



def policy_random():
    return random.randrange(0, total_actions)

def send_command(cmd):
    arduino.write(cmd.encode() + b'\n')
    time.sleep(0.1)
    response = arduino.readline().decode().strip()
    return response

def find_best_forward_action(row, col):
    return np.argmin(Q[:, row, col])

def find_best_backward_action(row, col):
    return np.argmax(Q[:, row, col])

try:
    while True:


        # Send command
        #print(send_command("SPEED:150"))
        #time.sleep(1)
        
        # Receive data
        if arduino.in_waiting > 0:
            line = arduino.readline().decode().strip()
            
            

            # if line.startswith("OK"):
            #     print(f"Response: {line} ....")

            if line.startswith("ACTION_REQUEST"):
                action = policy_random()
                send_command(f"Action:{action}")
                print(f"Sent action: {action}")

            if line.startswith("PROCESS_DATA:"):
                pattern = r'(\w+):(-?\d+\.?\d*)'
                matches = re.findall(pattern, line)
                data = {key: float(value) for key, value in matches}
                
                # Create variables for the data
                state_row, state_col = int(data['state_row']), int(data['state_col'])
                next_state_row, next_state_col = int(data['next_state_row']), int(data['next_state_col'])
                reward = data['step_reward']
                
                # Use variables in print statements
                print(f"State: ({state_row}, {state_col})")
                print(f"Reward: {reward}")
                print(f"Next State: ({next_state_row}, {next_state_col})")
                print("---")

                next_action = find_best_forward_action(next_state_row,next_state_col ) if forward else find_best_backward_action(next_state_row,next_state_col )

                #updating Q table
                Q[action][state_row][state_col] += learn_rate * (reward + discount_factor * Q[next_action][next_state_row][next_state_col] - Q[action][state_row][state_col]);


            if line.startswith("DEBUG"):
                print(f"Response: {line} ....")

        

except KeyboardInterrupt:
    print("Stopped")
    arduino.close()

Connected to: COM3
Is open: True
Sent action: 13
State: (7, 0)
Reward: 0.0
Next State: (7, 0)
---
Sent action: 1
State: (7, 0)
Reward: 0.0
Next State: (6, 0)
---
Sent action: 15
State: (6, 0)
Reward: 0.0
Next State: (6, 1)
---
Sent action: 16
State: (6, 1)
Reward: 0.0
Next State: (8, 1)
---
Sent action: 7
State: (8, 1)
Reward: 0.0
Next State: (9, 1)
---
Sent action: 5
State: (9, 1)
Reward: 0.0
Next State: (0, 1)
---
Sent action: 6
State: (0, 1)
Reward: 0.0
Next State: (0, 5)
---
Sent action: 12
State: (0, 5)
Reward: 0.0
Next State: (0, 6)
---
Sent action: 1
State: (0, 6)
Reward: 1.11
Next State: (5, 6)
---
Sent action: 7
State: (5, 6)
Reward: -1.72
Next State: (5, 1)
---
Sent action: 15
State: (5, 1)
Reward: 0.0
Next State: (0, 1)
---
Sent action: 11
State: (0, 1)
Reward: 0.0
Next State: (8, 1)
---
Sent action: 14
State: (8, 1)
Reward: 0.0
Next State: (4, 1)
---
Sent action: 7
State: (4, 1)
Reward: 0.0
Next State: (7, 1)
---
Sent action: 11
State: (7, 1)
Reward: 0.0
Next State: (0, 1)


SerialException: ClearCommError failed (OSError(22, 'The I/O operation has been aborted because of either a thread exit or an application request.', None, 995))